In [1]:
from ingest import load_faq_data

In [2]:
documents = load_faq_data()

In [3]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [4]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [5]:
documents = documents_llm

In [6]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.

""".strip()

In [9]:
structured_instructions = data_gen_instructions + """
Return ONLY a valid JSON object with the following structure:
{
    "questions": ["question1", "question2", "question3", "question4", "question5"]
}
Do not include any other text, markdown, or formatting outside the JSON.
"""

In [10]:
from dotenv import load_dotenv
from groq import Groq
import os
load_dotenv()
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [11]:
import json
user_prompt = json.dumps(doc)

In [12]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [13]:
messages = [
    {"role": "developer", "content": structured_instructions},
    {"role": "user", "content": user_prompt}
]

In [14]:
response = groq_client.chat.completions.create(
    model="openai/gpt-oss-120b",  
    messages=messages,
    response_format={"type": "json_object"},
    temperature=0.7,
)

In [15]:
# Parse the response
import json
result_text = response.choices[0].message.content
result_data = json.loads(result_text)
questions = result_data["questions"]
questions

['Is it still possible to enroll in the LLM Zoomcamp now?',
 'Can I get a certificate if I join late?',
 'What’s the deadline for submitting the final project to earn a cert?',
 'Do I have to finish the whole course to get the certificate?',
 'Are there any restrictions for new students who just found the course?']

In [16]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [17]:
from evaluation_utils import llm_structured

e:\Repos\DTC\llm\04-evaluation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
result, usage = llm_structured(
    groq_client,
    structured_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Is it still possible to sign up for the course now?', 'Can I join the class late and still get the certificate?', 'Do I need to submit my project to earn the certificate if I enroll late?', 'Are there any deadlines for project submission after I start the course?', 'What happens if I miss the project submission window—can I still get credit?']


In [19]:
usage

ResponseUsage(input_tokens=413, output_tokens=338, total_tokens=751)

In [20]:
from evaluation_utils import calc_price

In [21]:
calc_price(usage)

{'input_cost': 6.195e-05,
 'output_cost': 0.00020279999999999997,
 'total_cost': 0.00026474999999999996}

In [22]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Is it still possible to sign up for the course now?',
  'document': '74eb249bbf'},
 {'question': 'Can I join the class late and still get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit my project to earn the certificate if I enroll late?',
  'document': '74eb249bbf'},
 {'question': 'Are there any deadlines for project submission after I start the course?',
  'document': '74eb249bbf'},
 {'question': 'What happens if I miss the project submission window—can I still get credit?',
  'document': '74eb249bbf'}]

In [23]:
import pandas as pd

In [24]:
pd.DataFrame(records)

,question,document
0,Is it still possible to sign up for the course...,74eb249bbf
1,Can I join the class late and still get the ce...,74eb249bbf
2,Do I need to submit my project to earn the cer...,74eb249bbf
3,Are there any deadlines for project submission...,74eb249bbf
4,What happens if I miss the project submission ...,74eb249bbf


In [25]:
from evaluation_utils import llm_structured_retry

In [26]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        groq_client,
        structured_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [27]:
generate_ground_truth(doc)

([{'question': 'Can I enroll in the course now even though it already started?',
   'document': '74eb249bbf'},
  {'question': 'Is it still possible to earn a certificate if I join late?',
   'document': '74eb249bbf'},
  {'question': 'Do I have to submit a project to get the certificate?',
   'document': '74eb249bbf'},
  {'question': 'When is the deadline for project submissions for certification?',
   'document': '74eb249bbf'},
  {'question': 'What if I miss the submission deadline after joining late?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=413, output_tokens=403, total_tokens=816))

In [28]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

100%|██████████| 5/5 [00:06<00:00,  1.34s/it]


In [ ]:
from evaluation_utils import map_progress
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

In [ ]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

In [ ]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)